In [1]:
import pathlib

import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.metrics import accuracy_score, roc_auc_score

import yaml

import dataset_Oct22

In [2]:
OUTPUT_DIR = pathlib.Path("../output/ordinal_Oct22_models")
DATASET_CSV = pathlib.Path("../output/ordinal_Oct22_sequences_with_dca_score.csv")

In [3]:
ds = dataset_Oct22.Oct22DataSet(DATASET_CSV)
test_labels = ds.get_test_dataset().response.to_numpy()

In [4]:
# read in all the data from the output directory
model_info = {}
for filename in OUTPUT_DIR.glob("*.yml"):
    #print(filename)
    uuid = filename.stem
    with open(filename) as fh:
        mc = yaml.safe_load(fh)
    probs = np.loadtxt(OUTPUT_DIR / f"{uuid}.probs.txt")
    preds = np.loadtxt(OUTPUT_DIR / f"{uuid}.preds.txt")
    model_info[uuid] = {'config':mc, 'probs':probs, 'preds':preds}

In [5]:
test_labels_multiclass = np.eye(4)[test_labels]
test_labels_binary = np.eye(2)[test_labels.astype(bool).astype(int)]

In [6]:
model_extract = []
# extract info that needs to be plotted
for name, md in model_info.items():
    mc = md["config"]
    probs = md["probs"]
    preds = md["preds"]
    target = mc["target"]
    me =  {
            'model_name': mc["model_name"],
            'intercept': mc['design_matrix']["intercept"],
            'encoding': mc['design_matrix']["encoding"],
            'dca': mc['design_matrix']["dca"],
            'target': target,
            'accuracy': accuracy_score(test_labels, preds),
        }
    test_label_indicators = test_labels_multiclass
    if target == "binary":
        # make one dimensional arrays
        test_label_indicators = test_labels.astype(bool).astype(int)
        probs = probs[0, :]
    me["roc_score"] = roc_auc_score(test_label_indicators, probs, multi_class="ovr")
    me["unique_prediction"] = (len(np.unique(preds)) <= 1)
        
    model_extract.append(me)
model_extract_df = pd.DataFrame(model_extract)
model_extract_df

,model_name,intercept,encoding,dca,target,accuracy,roc_score,unique_prediction
0,SklearnLogisticRegression,True,one-hot,False,binary,0.507538,0.384845,True
1,SklearnLogisticRegression,True,one-hot,False,multiclass,0.547739,0.749731,False
2,SklearnRidgeClassifier,True,one-hot,False,multiclass,0.527638,0.698825,False
3,SklearnRidgeClassifier,False,one-hot,False,multiclass,0.527638,0.702736,False
4,SklearnRidgeClassifier,False,one-hot,False,binary,0.522613,0.350861,False
5,SklearnLogisticRegression,False,one-hot,False,binary,0.507538,0.373020,True
6,SklearnLogisticRegression,False,one-hot,False,multiclass,0.547739,0.749756,False
7,SklearnRidgeClassifier,True,one-hot,False,binary,0.522613,0.343628,False
